# 05 - Weights + Meta-dataset

1. Loads `val_probs_*.npy` for every base model and computes per-model validation accuracy.
2. Computes accuracy-normalized weights `w_i = acc_i / sum_j acc_j` and saves them to `artifacts/weights.json`.
3. Renders structured-token prompts for `train` (using OOF probs), `val`, and `test` and writes JSONL to `artifacts/meta_jsonl/{train,val,test}.jsonl`.

Train rows include the `<label>...</label>` completion (gold). Val rows include the gold completion too (for SFT eval). Test rows store the gold label in a separate `gold` field so the LLM never sees it during generation.

In [ ]:
import sys, os, json

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/vsfc_ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.VSFC_ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

In [ ]:
import numpy as np
from tm_research.VSFC_ensemble.utils_io import (
    load_vsfc_splits, load_probs, ARTIFACTS_DIR, META_JSONL_DIR
)
from tm_research.VSFC_ensemble.utils_stacking import (
    BASE_MODEL_NAMES, compute_weights, format_prompt, write_jsonl, weighted_average,
    build_meta_system_prompt, PROMPT_SCHEMA_VERSION,
)

train_df, val_df, test_df, label_map = load_vsfc_splits()
y_train = train_df['label'].map(label_map.label2id).to_numpy()
y_val = val_df['label'].map(label_map.label2id).to_numpy()
y_test = test_df['label'].map(label_map.label2id).to_numpy()
print('classes', label_map.class_names)
print('shapes', len(train_df), len(val_df), len(test_df))

In [ ]:
oof_probs = {m: load_probs(m, 'oof') for m in BASE_MODEL_NAMES}
val_probs = {m: load_probs(m, 'val') for m in BASE_MODEL_NAMES}
test_probs = {m: load_probs(m, 'test') for m in BASE_MODEL_NAMES}
for m in BASE_MODEL_NAMES:
    print(f'{m:8s}  oof={oof_probs[m].shape}  val={val_probs[m].shape}  test={test_probs[m].shape}')

## Weights from per-model validation accuracy

In [ ]:
val_acc = {m: float((val_probs[m].argmax(1) == y_val).mean()) for m in BASE_MODEL_NAMES}
weights = compute_weights(val_acc)
print('val accuracies:')
for m, a in val_acc.items():
    print(f'  {m:8s}  acc={a:.4f}  w={weights[m]:.4f}')

weighted_val = weighted_average(val_probs, weights)
weighted_val_acc = float((weighted_val.argmax(1) == y_val).mean())
print(f'weighted-avg val accuracy: {weighted_val_acc:.4f}')

with open(ARTIFACTS_DIR / 'weights.json', 'w', encoding='utf-8') as f:
    json.dump({
        'method': 'accuracy_normalized',
        'prompt_schema': PROMPT_SCHEMA_VERSION,
        'val_accuracy_per_model': val_acc,
        'weights': weights,
        'weighted_val_accuracy': weighted_val_acc,
        'system_prompt': build_meta_system_prompt(label_map, weights),
    }, f, ensure_ascii=False, indent=2)
print('saved', ARTIFACTS_DIR / 'weights.json')

## Render JSONL for train/val/test

In [ ]:
def build_rows(texts, probs_per_model, gold_labels=None, with_completion=True):
    rows = []
    n = len(texts)
    for i in range(n):
        per_model = {m: probs_per_model[m][i] for m in BASE_MODEL_NAMES}
        gold = gold_labels[i] if gold_labels is not None else None
        formatted = format_prompt(
            text=texts[i],
            probs_per_model=per_model,
            weights=weights,
            label_map=label_map,
            label=gold if with_completion else None,
        )
        row = {
            'idx': i,
            'text': texts[i],
            'prompt': formatted['prompt'],
            'completion': formatted['completion'],
        }
        if gold is not None:
            row['gold'] = gold
        rows.append(row)
    return rows

train_rows = build_rows(
    train_df['text'].tolist(), oof_probs,
    gold_labels=train_df['label'].tolist(), with_completion=True,
)
val_rows = build_rows(
    val_df['text'].tolist(), val_probs,
    gold_labels=val_df['label'].tolist(), with_completion=True,
)
test_rows = build_rows(
    test_df['text'].tolist(), test_probs,
    gold_labels=test_df['label'].tolist(), with_completion=False,
)

write_jsonl(train_rows, META_JSONL_DIR / 'train.jsonl')
write_jsonl(val_rows, META_JSONL_DIR / 'val.jsonl')
write_jsonl(test_rows, META_JSONL_DIR / 'test.jsonl')
print('wrote', len(train_rows), 'train,', len(val_rows), 'val,', len(test_rows), 'test')
print('first train prompt:\n', train_rows[0]['prompt'])
print('completion:', train_rows[0]['completion'])

In [ ]:
from tm_research.VSFC_ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()